# Task 8 – Collaborative Filtering Recommendation Model

## Objective
To recommend TED Talks based on user preferences and interaction history.

## 1. Load User Interaction Dataset

Load the simulated user interaction data created in Task 7.

In [1]:
import pandas as pd

interactions = pd.read_csv("../data/interaction/user_interactions.csv")

interactions.head()

,user_id,talk_id,rating
0,1,101,5
1,1,102,4
2,1,103,3
3,2,101,4
4,2,104,5


## 2. Inspect Interaction Data

Check the dataset structure and information.

In [2]:
interactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   user_id  15 non-null     int64
 1   talk_id  15 non-null     int64
 2   rating   15 non-null     int64
dtypes: int64(3)
memory usage: 492.0 bytes


In [3]:
interactions.columns

Index(['user_id', 'talk_id', 'rating'], dtype='str')

In [4]:
print("Dataset Shape:", interactions.shape)
print("Missing Values:\n", interactions.isnull().sum())

Dataset Shape: (15, 3)
Missing Values:
 user_id    0
talk_id    0
rating     0
dtype: int64


## 3. Create User-Item Interaction Matrix

Rows represent users, columns represent TED Talks, and values represent ratings.

In [5]:
user_item_matrix = interactions.pivot(
    index="user_id",
    columns="talk_id",
    values="rating"
)

user_item_matrix

talk_id,101,102,103,104,105
user_id,,,,,
1,5.0,4.0,3.0,NaN,NaN
2,4.0,NaN,NaN,5.0,3.0
3,NaN,5.0,4.0,NaN,2.0
4,3.0,NaN,5.0,4.0,NaN
5,NaN,4.0,NaN,5.0,3.0


## 4. Prepare Matrix for Cosine Similarity

Replace missing ratings with zero for similarity calculation. Zero represents missing interaction, not an actual rating.

In [6]:
user_item_filled = user_item_matrix.fillna(0)

user_item_filled

talk_id,101,102,103,104,105
user_id,,,,,
1,5.0,4.0,3.0,0.0,0.0
2,4.0,0.0,0.0,5.0,3.0
3,0.0,5.0,4.0,0.0,2.0
4,3.0,0.0,5.0,4.0,0.0
5,0.0,4.0,0.0,5.0,3.0


In [7]:
interactions["like_dislike"] = interactions["rating"].apply(
    lambda x: "Like" if x >= 4 else "Dislike"
)

interactions["interaction_type"] = "Watched"

interactions.head()

,user_id,talk_id,rating,like_dislike,interaction_type
0,1,101,5,Like,Watched
1,1,102,4,Like,Watched
2,1,103,3,Dislike,Watched
3,2,101,4,Like,Watched
4,2,104,5,Like,Watched


In [8]:
interactions.to_csv(
    "../data/interaction/user_interactions.csv",
    index=False
)

print("Updated interaction dataset saved successfully!")

Updated interaction dataset saved successfully!


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

user_item_matrix = interactions.pivot(
    index="user_id",
    columns="talk_id",
    values="rating"
)

user_item_filled = user_item_matrix.fillna(0)

similarity = cosine_similarity(user_item_filled)

user_similarity_matrix = pd.DataFrame(
    similarity,
    index=user_item_filled.index,
    columns=user_item_filled.index
)

user_similarity_matrix

user_id,1,2,3,4,5
user_id,,,,,
1,1.000000,0.400000,0.674619,0.600000,0.320000
2,0.400000,1.000000,0.126491,0.640000,0.680000
3,0.674619,0.126491,1.000000,0.421637,0.548128
4,0.600000,0.640000,0.421637,1.000000,0.400000
5,0.320000,0.680000,0.548128,0.400000,1.000000


## 6. Identify Similar Users

Select a user and find other users with similar preferences using cosine similarity scores.

In [10]:
selected_user = 1

similar_users = user_similarity_matrix[selected_user].drop(selected_user)

similar_users.sort_values(ascending=False)

user_id
3    0.674619
4    0.600000
2    0.400000
5    0.320000
Name: 1, dtype: float64

## 7. Develop Recommendation Function

Create a function that recommends unrated TED Talks based on ratings from similar users.

In [12]:
def recommend_for_user(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return "Invalid User ID"

    similar_users = user_similarity_matrix[user_id].drop(user_id)
    similar_users = similar_users[similar_users > 0].sort_values(ascending=False)

    rated_talks = user_item_matrix.loc[user_id].dropna().index
    unrated_talks = user_item_matrix.columns.difference(rated_talks)

    scores = {}

    for talk_id in unrated_talks:
        ratings = user_item_matrix.loc[similar_users.index, talk_id].dropna()
        common_users = similar_users.loc[ratings.index]

        if common_users.sum() > 0:
            scores[talk_id] = (
                ratings * common_users
            ).sum() / common_users.sum()

    recommendations = pd.Series(scores).sort_values(ascending=False)

    return recommendations.head(top_n)

## 8. Test the Recommendation Model

Test the recommendation function using at least three different users.

In [13]:
for user_id in [1, 2, 3]:
    print(f"\nRecommendations for User {user_id}:")
    print(recommend_for_user(user_id))


Recommendations for User 1:
104    4.545455
105    2.516270
dtype: float64

Recommendations for User 2:
103    4.205745
102    4.104842
dtype: float64

Recommendations for User 3:
104    4.615385
101    4.206897
dtype: float64


In [14]:
print(recommend_for_user(99))

Invalid User ID


## 9. Display TED Talk Titles

Connect recommended TED Talk IDs with the original TED Talks dataset and display their titles and recommendation scores.

In [15]:
talks = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

talks.head()

,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript,combined_text
0,1,averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,with the same humor and humanity he exuded in ...,"thank you so much, chris. and it's truly a gre...",averting the climate crisis ['alternative ener...
1,92,the best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['africa', 'asia', 'google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,you've never seen data presented like this. wi...,"about 10 years ago, i took on the task to teac...","the best stats you've ever seen ['africa', 'as..."
2,7,simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,new york times columnist david pogue takes aim...,"(music: ""the sound of silence,"" simon & garfun...","simplicity sells ['computers', 'entertainment'..."
3,53,greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['macarthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"in an emotionally charged talk, macarthur-winn...",if you're here today — and i'm very happy that...,"greening the ghetto ['macarthur grant', 'activ..."
4,66,do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,sir ken robinson makes an entertaining and pro...,good morning. how are you? (audience) good. it...,"do schools kill creativity? ['children', 'crea..."


We need to identify the correct ID column before matching recommendations with TED Talk titles.

In [16]:
talks.columns

Index(['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations',
       'about_speakers', 'views', 'recorded_date', 'published_date', 'event',
       'native_lang', 'available_lang', 'comments', 'duration', 'topics',
       'related_talks', 'url', 'description', 'transcript', 'combined_text'],
      dtype='str')

In [17]:
print("Interaction Talk IDs:")
print(sorted(interactions["talk_id"].unique()))

print("\nOriginal Dataset Talk IDs:")
print(talks["talk_id"].head())

Interaction Talk IDs:
[np.int64(101), np.int64(102), np.int64(103), np.int64(104), np.int64(105)]

Original Dataset Talk IDs:
0     1
1    92
2     7
3    53
4    66
Name: talk_id, dtype: int64


In [18]:
id_mapping = {
    101: 1,
    102: 92,
    103: 7,
    104: 53,
    105: 66
}

interactions["talk_id"] = interactions["talk_id"].map(id_mapping)

interactions.to_csv(
    "../data/interaction/user_interactions.csv",
    index=False
)

interactions.head()

,user_id,talk_id,rating,like_dislike,interaction_type
0,1,1,5,Like,Watched
1,1,92,4,Like,Watched
2,1,7,3,Dislike,Watched
3,2,1,4,Like,Watched
4,2,53,5,Like,Watched


In [19]:
user_item_matrix = interactions.pivot(
    index="user_id",
    columns="talk_id",
    values="rating"
)

user_item_filled = user_item_matrix.fillna(0)

similarity = cosine_similarity(user_item_filled)

user_similarity_matrix = pd.DataFrame(
    similarity,
    index=user_item_filled.index,
    columns=user_item_filled.index
)

user_similarity_matrix

user_id,1,2,3,4,5
user_id,,,,,
1,1.000000,0.400000,0.674619,0.600000,0.320000
2,0.400000,1.000000,0.126491,0.640000,0.680000
3,0.674619,0.126491,1.000000,0.421637,0.548128
4,0.600000,0.640000,0.421637,1.000000,0.400000
5,0.320000,0.680000,0.548128,0.400000,1.000000


In [20]:
recommendations = recommend_for_user(1)

recommendation_results = recommendations.reset_index()
recommendation_results.columns = ["talk_id", "predicted_rating"]

recommendation_results = recommendation_results.merge(
    talks[["talk_id", "title"]],
    on="talk_id",
    how="left"
)

recommendation_results[
    ["talk_id", "title", "predicted_rating"]
]

,talk_id,title,predicted_rating
0,53,greening the ghetto,4.545455
1,66,do schools kill creativity?,2.516270



## 10. Questions to Answer

### 1. What is Collaborative Filtering?
Collaborative Filtering recommends TED Talks based on the preferences and ratings of similar users.

### 2. How is Collaborative Filtering different from Content-Based Filtering?
Collaborative Filtering uses user preferences, while Content-Based Filtering uses TED Talk content such as titles, descriptions, and tags.

### 3. What is a User-Item Interaction Matrix?
It is a matrix where rows represent users, columns represent TED Talks, and values represent ratings.

### 4. Why do we calculate similarity between users?
To identify users who have similar interests and preferences.

### 5. What does a cosine similarity score represent?
It measures how similar two users' rating patterns are. A score closer to 1 indicates greater similarity.

### 6. Why should we exclude TED Talks already rated by a user?
To recommend TED Talks that the user has not rated yet.

### 7. What is a predicted rating?
It is an estimated rating calculated using ratings from similar users.

### 8. What happens when a new user has no interaction history?
The system cannot identify the user's preferences accurately.

### 9. What is the Cold-Start Problem?
It is the difficulty of generating recommendations for new users or items with little or no interaction data.

### 10. What are the limitations of User-Based Collaborative Filtering?
- Sparse interaction data.
- Cold-start problem.
- Difficulty finding similar users.
- Recommendations may be less accurate when user data is limited.


## 11. Observations

1. Cosine Similarity helps identify users with similar preferences.
2. Similar users' ratings are used to generate recommendations.
3. Different users may receive different recommendations.
4. User ratings influence predicted recommendation scores.
5. Already-rated TED Talks are excluded from recommendations.
6. Limited interaction data can affect recommendation accuracy.

## Conclusion

In this task, we developed a User-Based Collaborative Filtering model using simulated user interaction data. We calculated user similarity using Cosine Similarity and created a reusable recommendation function. We tested the model with different users and displayed recommended TED Talk titles with predicted ratings.